# 03 — The GQE solver

Running the CUDA-Q generative quantum eigensolver against a DMET
embedding.

**This notebook needs more setup than the others**: `quenais[cudaq]`, a
`gqe-for-qsci` checkout at commit `0a201ea`, and the source patch applied.
See `docs/gqe_integration.md`.

Everything before the training cell works without CUDA-Q, so you can read
the configuration parts regardless.

## 1. Is the checkout ready?

The runner refuses to launch against an unpatched checkout. Worth checking
first — the failure mode otherwise is a training run that completes
successfully and produces nothing parseable.

In [ ]:
from quenais.quantum import gqe_setup

repo = "../gqe-for-qsci"          # adjust to your checkout

problems = gqe_setup.verify_gqe_repo(repo)
if problems:
    print("Not ready:")
    for p in problems:
        print("  -", p)
    print(f"\nFix with: quenais-gqe-setup --repo {repo}")
else:
    print("Ready: correct commit, patch applied.")

## 2. Configure

The GQE settings map onto the external repo's Hydra config. Anything left
as `None` uses that repo's own default.

Two overrides are mandatory and always emitted:

- `molecule=dmet_embedding` — without it, `train.py` loads `n2.yaml` and
  trains on N₂ while reporting numbers that claim to be yours.
- `operator_pool.spec=dmet_excitation` — the stock pools rebuild the
  molecule from its geometry, and an embedding has none.

In [ ]:
from quenais import Config
from quenais.settings import GqeSettings

cfg = Config(
    molecule="LiH",
    basis="sto-3g",
    project_dir="./lih_run",
    quantum_solver="gqe",
    gqe=GqeSettings(
        repo_path=repo,
        max_iters=30,          # small, for a first run
        num_samples=100,
        batch_size=100,        # must equal num_samples
        ngates=20,
        cudaq_target="qpp-cpu",
    ),
)
cfg.validate().make_dirs().load_geometry()

for override in cfg.gqe.hydra_overrides(cfg.step2_file):
    print(" ", override)

## 3. Which pool, and why it matters

`dmet_excitation` accumulates all the Pauli terms of an excitation into a
single operator. Particle-number conservation is a property of that
**sum** — no individual Pauli word conserves it alone.

`dmet_pauli_evolution` appends each term separately, so it cannot conserve
electron number however its flags are set. On ScH roughly half of every
sample was discarded as symmetry-violating.

In [ ]:
from quenais.settings.gqe import DMET_POOL_SPECS, OPERATOR_POOL_SPECS

print("registered by the patch :", OPERATOR_POOL_SPECS)
print("usable with an embedding:", DMET_POOL_SPECS)

# The stock pools are rejected at config time rather than 20 minutes into a run.
try:
    GqeSettings(operator_pool_spec="excitation").validate()
except ValueError as exc:
    print("\n", exc)

## 4. Train

Runs the external `train.py` as a subprocess, streaming its output. The
runner verifies the patch, checks the step 2 pickle belongs to this
molecule, generates the Hydra molecule config, and puts the shim directory
on `PYTHONPATH`.

LiH converges to the embedded CASCI energy essentially exactly, so it is a
clean pass/fail.

In [ ]:
from quenais.quantum import gqe_runner

result = gqe_runner.main(cfg, force=True)
print(result)

## 5. Read the training log

The `[epoch N] {...}` lines come from the `train_pipeline.py` patch hunk.
If there are none, the checkout is unpatched — training will have looked
entirely successful.

In [ ]:
from quenais.visualization import plots

rows = plots.parse_gqe_log(cfg.gqe_log_file)
print(f"{len(rows)} epochs")

if rows:
    key = "Global-refined(best_so_far)/energy - R-CASCI"
    x, y = plots._col(rows, key)

    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(x, y)
    ax.axhline(1.6e-3, color="gray", ls="--", label="chemical accuracy")
    ax.set_yscale("log")
    ax.set_xlabel("epoch")
    ax.set_ylabel("|error vs CASCI| (Ha)")
    ax.legend()

## 6. Figures and summary

In [ ]:
plots.main(cfg)

## Sampling capacity is the limit on larger systems

GQE's accuracy is bounded by how much of the determinant space it can
reach. ScH (22 qubits, ~109k determinants) showed this clearly:

| settings | error vs embedded CASCI |
|---|---|
| ngates=10, samples=10 | 60.4 mHa — stalled at HF |
| ngates=20, samples=100 | 36.8 mHa |
| ngates=40, samples=100 | 24.1 mHa — subspace hit the cap |

LiH and N₂ (4 embedding orbitals) converge exactly, so this is a capacity
limit on larger systems rather than a correctness problem. See
`docs/limitations.md`.